# Gold layer — Bank Reconciliation

**Data product owner:** Finance. **Grain:** one reconciliation result per ERPNext source instance and bank GL entry within the demo run.

1. Read the three Silver Delta tables for the existing demo run.
2. Resolve the baseline bank account and identify eligible statement transactions.
3. Apply one-to-one matching, report missing coverage and preserve evidence.
4. Publish and verify the Finance-owned Gold Delta table.

### Setup

Run Silver successfully first. Use the same Python 3.11 kernel, JDK 17, PySpark 3.5.6 and the local Git-ignored `data-platform/.env`. Add the following settings to your template and fill them in locally:

```dotenv
GOLD_BUCKET=gold
GOLD_MATCH_WINDOW_DAYS=0
GOLD_DEMO_BANK_ACCOUNT_ID=DEMO-001
GOLD_DEMO_STATEMENT_DIRECTION=withdrawal
GOLD_DEMO_CURRENCY=INR
GOLD_DEMO_COVERAGE_START=
GOLD_DEMO_COVERAGE_END=
```

Create the Gold bucket and use credentials with Silver read and Gold write access. Launch from within the repository. Review the demo statement and enter both coverage dates (inclusive ISO dates) to attest that the statement covers the entire matching window for that account. In the seeded same-day case with a zero-day window, both dates are the seeded posting date. Blank dates produce `INCOMPLETE_COVERAGE`. Silver stores unsigned statement amounts and omits currency and coverage metadata. The settings above are explicit assumptions for the synthetic fixture and are recorded in Gold. Use source-supplied metadata before applying this rule to other statements.

### Scenario checks

| Scenario | Expected status |
|---|---|
| One eligible line and complete coverage | MATCHED |
| No eligible line and complete coverage | UNMATCHED |
| Two eligible lines or one line eligible for two GL entries | AMBIGUOUS |
| Missing coverage for the matching window | INCOMPLETE_COVERAGE |
| Missing or multiple mappings | MISSING_MAPPING or AMBIGUOUS_MAPPING |

The notebook runs these checks before writing. Gold publishes the outcome, reason, rule version, and evidence keys for later read-only consumption.


In [ ]:
import json
import os
import sys
from decimal import Decimal
from pathlib import Path
from urllib.parse import urlsplit
from uuid import uuid4

from dotenv import dotenv_values
from pyspark.sql import SparkSession, functions as F

# Support launching Jupyter from either the repository root or the notebook folder.
cwd = Path.cwd().resolve()
candidates = [parent if parent.name == "data-platform" else parent / "data-platform"
              for parent in (cwd, *cwd.parents)]
env_path = next((folder / ".env" for folder in candidates if (folder / ".env").is_file()), None)
if env_path is None:
    raise FileNotFoundError("Create data-platform/.env and launch from the repository.")
cfg = dotenv_values(env_path, interpolate=False)

def required(name):
    value = cfg.get(name)
    if not value:
        raise ValueError(f"Set {name} in data-platform/.env")
    return value

endpoint = cfg.get("MINIO_ENDPOINT") or "http://192.168.29.250:9000"
bucket = cfg.get("BRONZE_BUCKET") or "bronze"
os.environ["AWS_ACCESS_KEY_ID"] = required("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = required("AWS_SECRET_ACCESS_KEY")
os.environ["PYSPARK_PYTHON"] = sys.executable

spark = (
    SparkSession.builder.master("local[2]").appName("gold-bank-reconciliation")
    .config("spark.jars.packages",
            "io.delta:delta-spark_2.12:3.3.3,org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.hadoop.fs.s3a.endpoint", endpoint)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", str(urlsplit(endpoint).scheme == "https").lower())
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)

In [ ]:
from collections import Counter, defaultdict
from datetime import date, timedelta

RULE_VERSION = "demo-exact-amount-direction-date-v1"

def reconcile(gl_rows, statements, mappings, account, direction, coverage, window):
    # Index statement rows by account and exact decimal amount.
    index = defaultdict(list)
    for statement in statements:
        index[(statement.bank_account_id, statement.amount)].append(statement)
    accounts = defaultdict(set)
    for mapping in mappings:
        if mapping.scenario == "baseline":
            accounts[mapping.gl_id].add(mapping.bank_account_id)

    proposals = []
    claims = Counter()
    for gl in gl_rows:
        choices = accounts[gl.gl_id]
        bank = next(iter(choices)) if len(choices) == 1 else None
        start, end = gl.posting_date - window, gl.posting_date + window
        complete = bool(bank == account and coverage and coverage[0] <= start and coverage[1] >= end)
        expected_direction = "deposit" if gl.debit > 0 else "withdrawal"
        amount = gl.debit if gl.debit > 0 else gl.credit
        candidates = []
        if bank == account and expected_direction == direction:
            candidates = sorted(
                (s for s in index[(bank, amount)] if start <= s.posting_date <= end),
                key=lambda s: (s.raw_uri, s.transaction_id),
            )
        claims.update((s.raw_uri, s.transaction_id) for s in candidates)
        proposals.append((gl, choices, bank, amount, complete, candidates))

    for gl, choices, bank, amount, complete, candidates in proposals:
        if len(choices) != 1:
            status = "AMBIGUOUS_MAPPING" if choices else "MISSING_MAPPING"
            reason = "Expected exactly one baseline bank account mapping"
        elif not complete:
            status, reason = "INCOMPLETE_COVERAGE", "Demo coverage does not span the matching window"
        elif not candidates:
            status, reason = "UNMATCHED", "No statement row meets the matching rule"
        elif len(candidates) > 1 or claims[(candidates[0].raw_uri, candidates[0].transaction_id)] > 1:
            status, reason = "AMBIGUOUS", "Multiple candidates or a statement row shared by GL entries"
        else:
            status, reason = "MATCHED", "Unique candidate by account, amount, direction and date"
        yield (gl.source_instance, gl.gl_id, gl.company, gl.ledger_account, bank,
               gl.posting_date, amount, status, reason, complete, gl.raw_uri,
               [(s.transaction_id, s.raw_uri) for s in candidates])


In [ ]:
from types import SimpleNamespace as Record

# Exercise the same rule used by the live Silver read.
day = date(2026, 11, 2)
gl = Record(source_instance="demo", gl_id="GL1", company="Demo", ledger_account="Bank",
            posting_date=day, credit=Decimal("10000"), debit=Decimal("0"), raw_uri="gl")
line = Record(bank_account_id="DEMO-001", amount=Decimal("10000"), posting_date=day,
              raw_uri="statement", transaction_id="TX1")
mapping = Record(scenario="baseline", gl_id="GL1", bank_account_id="DEMO-001")
other_line = Record(**(vars(line) | {"transaction_id": "TX2"}))
other_gl = Record(**(vars(gl) | {"gl_id": "GL2"}))
other_mapping = Record(**(vars(mapping) | {"gl_id": "GL2"}))
extra_mapping = Record(**(vars(mapping) | {"bank_account_id": "DEMO-002"}))
cases = [
    ([gl], [line], [mapping], (day, day), ["MATCHED"]),
    ([gl], [], [mapping], (day, day), ["UNMATCHED"]),
    ([gl], [line, other_line], [mapping], (day, day), ["AMBIGUOUS"]),
    ([gl, other_gl], [line], [mapping, other_mapping], (day, day), ["AMBIGUOUS", "AMBIGUOUS"]),
    ([gl], [line], [mapping], None, ["INCOMPLETE_COVERAGE"]),
    ([gl], [line], [], (day, day), ["MISSING_MAPPING"]),
    ([gl], [line], [mapping, extra_mapping], (day, day), ["AMBIGUOUS_MAPPING"]),
]
for gs, ss, ms, period, expected in cases:
    actual = [r[7] for r in reconcile(gs, ss, ms, "DEMO-001", "withdrawal", period, timedelta(0))]
    if actual != expected:
        raise AssertionError((expected, actual))
print(f"PASS: {len(cases)} Gold rule scenarios")


In [ ]:
import re

run = required("DEMO_RUN_ID")
if not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", run):
    raise ValueError("Invalid DEMO_RUN_ID")
silver_bucket, gold_bucket = required("SILVER_BUCKET"), required("GOLD_BUCKET")
if gold_bucket in {silver_bucket, bucket}:
    raise ValueError("Choose a dedicated Gold bucket.")
account = required("GOLD_DEMO_BANK_ACCOUNT_ID")
direction = required("GOLD_DEMO_STATEMENT_DIRECTION")
if direction not in {"withdrawal", "deposit"}:
    raise ValueError("Choose withdrawal or deposit.")
currency = required("GOLD_DEMO_CURRENCY")
if currency != "INR":
    raise ValueError("This fixture uses INR.")
days = int(cfg.get("GOLD_MATCH_WINDOW_DAYS") or 0)
if not 0 <= days <= 30:
    raise ValueError("Match window must be between 0 and 30 calendar days.")
dates = [cfg.get(f"GOLD_DEMO_COVERAGE_{bound}") for bound in ("START", "END")]
if any(dates) and not all(dates):
    raise ValueError("Provide both coverage dates or leave both blank.")
coverage = tuple(date.fromisoformat(value) for value in dates) if all(dates) else None
if coverage and coverage[0] > coverage[1]:
    raise ValueError("Coverage start is after coverage end.")

root = f"s3a://{silver_bucket}/tables/silver-demo/{run}"
def read_demo(name):
    # This driver-side implementation is bounded to the seeded POC.
    rows = spark.read.format("delta").load(f"{root}/{name}").take(1001)
    if len(rows) > 1000:
        raise ValueError(f"{name} exceeds the demo limit of 1,000 rows.")
    return rows

gl_rows = read_demo("erpnext_gl")
keys = {(g.source_instance, g.gl_id) for g in gl_rows}
if not keys or len(keys) != len(gl_rows) or len({g.gl_id for g in gl_rows}) != len(keys):
    raise ValueError("Expected GL records with unique IDs across this demo run.")
statements = read_demo("statement_transactions")
mappings = read_demo("account_mapping_checks")
# The current Silver package records failed mapping scenarios in exceptions.
# Gold consumes baseline successes. Synthetic mapping failures are tested above.
results = list(reconcile(gl_rows, statements, mappings, account, direction,
                         coverage, timedelta(days=days)))
records = [r + (run, RULE_VERSION, currency, direction, days,
                *(coverage or (None, None))) for r in results]
schema = (
    "source_instance STRING, gl_id STRING, company STRING, ledger_account STRING, "
    "bank_account_id STRING, posting_date DATE, amount DECIMAL(18,2), status STRING, "
    "reason STRING, coverage_complete BOOLEAN, gl_raw_uri STRING, "
    "candidates ARRAY<STRUCT<transaction_id:STRING,raw_uri:STRING>>, "
    "demo_run_id STRING, rule_version STRING, currency STRING, statement_direction STRING, "
    "window_days INT, coverage_start DATE, coverage_end DATE"
)
frame = spark.createDataFrame(records, schema)
uri = f"s3a://{gold_bucket}/tables/bank_reconciliation/{run}"
# Replace only the selected demo run snapshot, so reruns do not append results.
frame.write.format("delta").mode("overwrite").save(uri)
actual = spark.read.format("delta").load(uri)
if (actual.schema.simpleString() != frame.schema.simpleString()
    or actual.exceptAll(frame).take(1) or frame.exceptAll(actual).take(1)):
    raise RuntimeError("Gold read-back differs from the published snapshot.")
actual.groupBy("status").count().orderBy("status").show()
print(f"PASS: verified {len(records)} Gold results at {uri}")
